# 10 · CLIP: imágenes y texto
*Dos encoders, una matriz de similitud y clasificar con frases*

Notebook del episodio 10 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/clip-imagenes-y-texto)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** load_digits: 1.200 train / 597 test, 4 plantillas de texto ('un siete', 'el número 7', 'dígito siete', 'un 7 escrito a mano') → batch de 10 pares, un dígito por clase (sin falsos negativos) → encoders: MLP 64→128→32 y bolsa de palabras 64→32, normalizados → S = Î T̂ᵀ (10×10) → InfoNCE simétrica con 1/τ = 14,3 inicial: pérdida al azar 3,206 (> ln 10 = 2,303) → 1.500 pasos de Adam: pérdida 0,019, 1/τ = 15,7 → zero-shot en test: 'un {p}' 95,5 %, 'el número {p}' (combinación nunca vista) 95,3 %, 'una foto de un {p}' 96,5 % → softmax con τ = 1 (uno = 20 %) vs. 1/τ = 15,7 (≈ 100 %) → coseno medio por clase: diagonal 0,65–0,83 → paper CLIP: 400 M pares, τ inicial 0,07, batch 32.768, zero-shot ImageNet 76,2 % = ResNet-50, 'A photo of a {label}.' +1,3 % → mapa de la serie y Resumiendo

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio diez, el último: CLIP, contrastive language image pretraining. Hasta acá, cada red que entrenamos tenía diez salidas fijas, una por clase. Hoy vamos a juntar una red que mira imágenes con otra que lee texto, y vamos a clasificar escribiendo frases.*

### Clasificar sin una capa de clases

- Hasta ahora: la última capa tiene una salida por clase
- CLIP: un vector para la imagen, otro para el texto
- Entrenar: el par correcto cerca, los demás lejos
- Clasificar: comparar la imagen con una frase por clase

El nombre, traducido: preentrenamiento contrastivo de lenguaje e imagen. Todas las redes de la serie terminan en una capa lineal con una salida por clase; si querés una clase nueva, reentrenás. CLIP cambia la pregunta. Una red convierte la imagen en un vector, otra convierte un texto en un vector del mismo tamaño. Se entrenan juntas para que la imagen y su descripción queden cerca, y las descripciones de otras imágenes, lejos. Y para clasificar, escribís una frase por clase y elegís la más parecida a la imagen. Las clases pasan a ser texto.

## Parte 1 · Dos encoders, un espacio

Un juguete que corre en segundos: los dígitos de ocho por ocho de scikit-learn y frases inventadas.

### Dos caminos que terminan en el mismo espacio

El esquema. Arriba, la imagen entra a un encoder de imagen y sale un vector I de treinta y dos números. Abajo, una frase entra a un encoder de texto y sale un vector T del mismo tamaño. Y los dos se comparan con el coseno.

dígito 8×8 → encoder de imagen → vector I (32) → «un siete» → encoder de texto → vector T (32) → coseno I·T

### Los pares imagen-texto

Los datos. Mil setecientos noventa y siete dígitos escritos a mano, de ocho por ocho píxeles: mil doscientos para entrenar y el resto para test. Para cada imagen inventamos una descripción con una de cuatro plantillas: un siete, el número siete con cifra, dígito siete, un siete escrito a mano. Y cada batch tiene diez pares: un dígito de cada clase, con su frase. El batch es de diez por sesenta y cuatro. Un dígito por clase, a propósito: ya vamos a ver por qué.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_pares.py
__file__ = 'snippets/ep10_pares.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, random
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _clip import datos, frase, PLANTILLAS, batch_un_digito_por_clase

In [3]:
Xtr, ytr, Xte, yte = datos()          # load_digits: 8×8 = 64 píxeles
print("imágenes:", tuple(Xtr.shape), "train,", len(yte), "test")
for p in PLANTILLAS:                  # 4 plantillas de texto
    print(f"  {p!r:26} → {frase(7, p)!r}")

rng = random.Random(0)
imgs, textos = batch_un_digito_por_clase(Xtr, ytr, rng)
print("batch:", tuple(imgs.shape), textos[:3])

imágenes: (1200, 64) train, 597 test
  'un {p}'                   → 'un siete'
  'el número {c}'            → 'el número 7'
  'dígito {p}'               → 'dígito siete'
  'un {c} escrito a mano'    → 'un 7 escrito a mano'
batch: (10, 64) ['dígito cero', 'un 1 escrito a mano', 'dígito dos']


### Los dos encoders

El encoder de imagen es un MLP, multilayer perceptron: sesenta y cuatro píxeles, una capa de ciento veintiocho neuronas con ReLU, rectified linear unit, y treinta y dos salidas. La salida se normaliza a largo uno. El de texto busca un embedding de sesenta y cuatro números por palabra y los promedia: una bolsa de palabras, que no mira el orden. Y una capa lineal lo lleva a treinta y dos. El vocabulario tiene treinta palabras, los dos vectores miden uno, y al azar el coseno da menos cero coma cero cinco.

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_encoders.py
__file__ = 'snippets/ep10_encoders.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys
import torch
import torch.nn as nn
import torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _clip import VOCAB, tokens, datos
torch.manual_seed(0)
Xtr, _, _, _ = datos()

In [5]:
class EncImagen(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.red = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, d))
    def forward(self, x):                       # x: N × 64 píxeles
        return F.normalize(self.red(x), dim=-1)  # largo 1

class EncTexto(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.emb = nn.EmbeddingBag(len(VOCAB), 64, mode="mean")  # promedio
        self.proy = nn.Linear(64, d)
    def forward(self, frases):
        return F.normalize(self.proy(self.emb(*tokens(frases))), dim=-1)

I, T = EncImagen()(Xtr[:2]), EncTexto()(["un siete", "dígito siete"])
print("vocabulario:", len(VOCAB), "palabras | I:", tuple(I.shape), "T:", tuple(T.shape))
print("largos:", I.norm(dim=-1).tolist(), "| cos(I0, T0) al azar:", round((I[0] @ T[0]).item(), 2))

vocabulario: 30 palabras | I: (2, 32) T: (2, 32)
largos: [1.0, 1.0] | cos(I0, T0) al azar: -0.05


### Coseno: cuánto apuntan para el mismo lado

$$
\hat I = \frac{I}{\lVert I \rVert},\quad \hat T = \frac{T}{\lVert T \rVert}
$$
<small>I y T: vectores de 32 números; ‖·‖ es el largo</small>

$$
\cos(I, T) = \hat I \cdot \hat T = \textstyle\sum_{k=1}^{32} \hat I_k\, \hat T_k
$$
<small>multiplicar coordenada a coordenada y sumar</small>

$$
S = \hat I\, \hat T^{\top} \in \mathbb{R}^{N \times N}
$$
<small>todas las imágenes contra todos los textos de una vez</small>

I es el vector de la imagen y T el del texto, treinta y dos números cada uno. Dividirlos por su largo, la norma, los deja de largo uno: I sombrero y T sombrero. El coseno entre los dos es su producto punto: multiplicar coordenada a coordenada y sumar. En el dibujo, cinco productos que suman cero coma ocho: apuntan casi para el mismo lado. Con un batch de N pares apilamos los vectores en matrices, y una sola multiplicación da todos los cosenos: la matriz S, de N por N. En el dibujo, la fila uno de I sombrero por la columna dos de T sombrero transpuesta llena la celda uno dos: imagen uno contra texto dos.

## Parte 2 · La pérdida contrastiva

En la matriz S, la diagonal son los pares correctos. Todo lo demás son pares cruzados.

### InfoNCE en ocho líneas

La pérdida se llama InfoNCE, info noise contrastive estimation. Los logits son los cosenos divididos por una temperatura tau, un escalar; en el código guardamos el logaritmo de uno sobre tau, para que siempre sea positivo. El objetivo de la fila i es la columna i: la diagonal. Entropía cruzada por filas: cada imagen tiene que elegir su texto entre los diez. Y por columnas, con la matriz transpuesta: cada texto tiene que elegir su imagen. Se promedian las dos: es simétrica. Con los encoders al azar, la pérdida da tres coma dos uno. El azar puro sería logaritmo de diez, dos coma tres: da más porque uno sobre tau, catorce coma tres, amplifica cosenos que todavía son ruido.

**Cada línea, su término:**

- `logits = log_escala.exp() * I @ T.T` → $L = S / \tau$ — cosenos divididos por la temperatura τ
- `obj = torch.arange(len(I))` → $y_i = i$ — la respuesta correcta es la diagonal
- `perd_i = F.cross_entropy(logits, obj)` → $\ell_I = -\tfrac{1}{N}\sum_i \log \mathrm{softmax}(L_{i,:})_i$ — cada imagen elige su texto
- `perd_t = F.cross_entropy(logits.T, obj)` → $\ell = \tfrac{1}{2}(\ell_I + \ell_T)$ — y cada texto elige su imagen

In [6]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_infonce.py
__file__ = 'snippets/ep10_infonce.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, random
import torch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from _clip import datos, EncImagen, EncTexto, batch_un_digito_por_clase
import torch.nn.functional as F
torch.manual_seed(0)
Xtr, ytr, _, _ = datos()
imgs, textos = batch_un_digito_por_clase(Xtr, ytr, random.Random(0))
enc_i, enc_t = EncImagen(), EncTexto()

In [7]:
def infonce(img, txt, log_escala):
    I = enc_i(img)                     # N × 32, cada fila con norma 1
    T = enc_t(txt)                     # N × 32
    logits = log_escala.exp() * I @ T.T   # N × N: cosenos / τ
    obj = torch.arange(len(I))         # el par correcto: la diagonal
    perd_i = F.cross_entropy(logits, obj)    # cada imagen elige texto
    perd_t = F.cross_entropy(logits.T, obj)  # cada texto elige imagen
    return (perd_i + perd_t) / 2, logits

log_escala = torch.tensor(1 / 0.07).log()    # τ = 0,07 como en CLIP
perd, logits = infonce(imgs, textos, log_escala)
print("logits:", tuple(logits.shape), "escala 1/τ =",
      round(log_escala.exp().item(), 1))
print("pérdida al azar:", round(perd.item(), 3),
      " ln 10 =", round(torch.tensor(10.).log().item(), 3))

logits: (10, 10) escala 1/τ = 14.3
pérdida al azar: 3.206  ln 10 = 2.303


### La temperatura afila el softmax

$$
p_{ik} = \frac{e^{S_{ik}/\tau}}{\sum_{j} e^{S_{ij}/\tau}}
$$
<small>softmax de una fila: probabilidades que suman 1</small>

$$
\tau = 1:\ \ p_{i,\text{uno}} \text{ apenas gana}
$$
<small>cosenos entre −1 y 1: el softmax queda chato</small>

$$
1/\tau = 15{,}7:\ \ p_{i,\text{uno}} \approx 1
$$
<small>la misma fila multiplicada por 15,7</small>

¿Para qué la temperatura? El softmax convierte una fila de números en probabilidades: exponencial de cada uno, dividido por la suma de todas. En el dibujo, una imagen de test que es un uno, contra las diez frases: los cosenos reales del modelo entrenado. El de uno es cero coma setenta y seis, el siguiente cero coma diecisiete. Con tau igual a uno, el softmax sobre números entre menos uno y uno queda chato: el uno gana, pero con apenas un veinte por ciento de la probabilidad. Dividir por tau, o sea multiplicar por quince coma siete, estira las diferencias, y la probabilidad del uno sube a prácticamente uno. CLIP aprende tau como un parámetro más.

> **Radford et al., 2021 — Learning Transferable Visual Models From Natural Language Supervision** · [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)
>
> **400 M pares** · «a dataset of 400 million (image, text) pairs collected from the internet»; τ «initialized to the equivalent of 0.07» y minibatch de 32.768.

El paper de CLIP. Entrenaron con cuatrocientos millones de pares de imagen y texto sacados de internet, con la misma pérdida simétrica que acabamos de escribir. La temperatura arranca en el equivalente a cero coma cero siete, como en nuestro snippet, y la recortan para que nunca multiplique los logits por más de cien. Y el batch es de treinta y dos mil setecientos sesenta y ocho pares: cuantos más pares cruzados, más difícil y más informativa es cada elección.

## Parte 3 · Entrenar los dos encoders

Un solo optimizador para las dos redes y la temperatura.

### El loop de entrenamiento

La escala ahora es un parámetro: el gradiente también la mueve. Un solo optimizador Adam, adaptive moment estimation, para los pesos de las dos redes y la escala. En cada paso, un batch nuevo de diez pares y la pérdida simétrica. Y el recorte de CLIP: uno sobre tau nunca pasa de cien. La pérdida baja de tres coma dos a cero coma cuatro en cien pasos, y queda cerca de cero. La escala apenas se movió: de catorce coma dos a quince coma siete.

In [8]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_entrenar.py
__file__ = 'snippets/ep10_entrenar.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, random
import torch
import torch.nn.functional as F
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _clip import datos, EncImagen, EncTexto, batch_un_digito_por_clase
torch.manual_seed(0)
Xtr, ytr, _, _ = datos()
enc_i, enc_t = EncImagen(), EncTexto()
rng = random.Random(0)


def infonce(I, T, log_escala):
    logits = log_escala.exp() * I @ T.T
    obj = torch.arange(len(I))
    return (F.cross_entropy(logits, obj) + F.cross_entropy(logits.T, obj)) / 2

In [9]:
log_escala = torch.nn.Parameter(torch.tensor(1 / 0.07).log())
params = [*enc_i.parameters(), *enc_t.parameters(), log_escala]
opt = torch.optim.Adam(params, lr=3e-3)
for paso in range(1, 1501):
    imgs, textos = batch_un_digito_por_clase(Xtr, ytr, rng)
    perd = infonce(enc_i(imgs), enc_t(textos), log_escala)
    opt.zero_grad(); perd.backward(); opt.step()
    with torch.no_grad():                 # CLIP: 1/τ nunca pasa de 100
        log_escala.clamp_(max=torch.tensor(100.).log())
    if paso in (1, 100, 500, 1500):
        print(f"paso {paso:4}  pérdida {perd.item():.3f}"
              f"  1/τ = {log_escala.exp().item():.1f}")

paso    1  pérdida 3.206  1/τ = 14.2
paso  100  pérdida 0.398  1/τ = 14.2


paso  500  pérdida 0.003  1/τ = 14.9


paso 1500  pérdida 0.019  1/τ = 15.7


In [10]:
# Verificación extra (corrió junto con el video)
torch.save({"img": enc_i.state_dict(), "txt": enc_t.state_dict(), "log_escala": log_escala.detach()},
           os.path.join(AQUI, "..", "datos", "clip_ep10.pt"))

### El batch define qué es un negativo

En nuestro batch hay un dígito de cada clase, así que todo lo que está fuera de la diagonal es de verdad otra clase. En CLIP no hay clases: si en un batch caen dos fotos de perros, la pérdida trata la descripción del otro perro como incorrecta. Son falsos negativos. Con treinta y dos mil pares y textos tan variados pasa poco, y el ruido se promedia. Con diez dígitos al azar pasaría todo el tiempo, por eso armamos el batch así.

## Parte 4 · Zero-shot

Clasificar sin entrenar un clasificador: las clases son frases.

### Un clasificador escrito en texto

$$
w_k = \mathrm{enc}_T(\text{el número } k),\ k = 0..9
$$
<small>una frase por clase → 10 vectores de 32</small>

$$
\hat y = \arg\max_k\; \hat I \cdot w_k
$$
<small>la clase cuya frase está más cerca</small>

El truco del zero-shot, clasificar sin haber entrenado un clasificador. Escribimos una frase por clase, el número cero hasta el número nueve, y las pasamos por el encoder de texto. En el dibujo, cada frase se convierte en una fila de una matriz W de diez por treinta y dos. Esa matriz hace de última capa lineal, pero escrita con texto en vez de entrenada. Para una imagen, la multiplicamos por W transpuesta: diez cosenos, uno por frase. En el dibujo, el vector de la imagen recorre la columna de una frase y llena su celda. Gana la frase con el coseno más alto.

### Tres formas de escribir las clases

Las diez frases salen de una plantilla. Quinientas noventa y siete imágenes de test contra las diez frases: una matriz de cosenos, y el máximo de cada fila es la predicción. Probamos tres plantillas. Un siete, que se usó para entrenar: noventa y cinco coma cinco por ciento. El número siete escrito con palabra: noventa y cinco coma tres. Esa combinación nunca apareció, porque en el entrenamiento número solo iba con cifras. Y una foto de un siete: noventa y seis coma cinco, aunque una, foto y de nunca se entrenaron. Sus embeddings son ruido fijo que se promedia, y la palabra de la clase manda.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep10_zeroshot.py
__file__ = 'snippets/ep10_zeroshot.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import os, sys, json
import torch
AQUI = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, AQUI)
from _clip import datos, EncImagen, EncTexto, PALABRAS
_, _, Xte, yte = datos()
ck = torch.load(os.path.join(AQUI, "..", "datos", "clip_ep10.pt"))
enc_i, enc_t = EncImagen(), EncTexto()
enc_i.load_state_dict(ck["img"]); enc_t.load_state_dict(ck["txt"])
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [12]:
def zero_shot(plantilla):
    clases = [plantilla.format(p=p) for p in PALABRAS]  # 10 frases
    T = enc_t(clases)                  # 10 × 32: el "clasificador"
    sims = enc_i(Xte) @ T.T            # 597 × 10 cosenos
    return (sims.argmax(1) == yte).float().mean().item(), sims

for plantilla in ["un {p}", "el número {p}", "una foto de un {p}"]:
    acc, sims = zero_shot(plantilla)
    print(f"{plantilla!r:22} precisión test: {acc:.1%}")

'un {p}'               precisión test: 95.5%
'el número {p}'        precisión test: 95.3%
'una foto de un {p}'   precisión test: 96.5%


In [13]:
# Verificación extra (corrió junto con el video)
_, sims = zero_shot("el número {p}")
fila = (sims[0] * ck["log_escala"].exp()).softmax(-1)
json.dump({"y0": int(yte[0]), "cos0": [round(v, 2) for v in sims[0].tolist()],
           "prob0": [round(v, 3) for v in fila.tolist()],
           "media": [[round(v, 2) for v in sims[yte == d].mean(0).tolist()] for d in range(10)]},
          open(os.path.join(AQUI, "..", "datos", "ep10_fig.json"), "w"))

### Coseno promedio por dígito y por frase

La misma información, promediada por clase. Cada fila son las imágenes de test de un dígito; cada columna, la frase el número k. La diagonal brilla: entre cero coma sesenta y cinco y cero coma ochenta y tres. Fuera de la diagonal casi todo está cerca de cero o es negativo, en rojo. Las excepciones se parecen a errores humanos: las imágenes de nueve tienen cero coma treinta y nueve con la frase del tres y cero coma treinta y seis con la del ocho. Y entre seis y nueve hay un menos cero coma setenta y tres: el modelo aprendió a separarlos con fuerza.

### Lo que este juguete no puede

- Una clase cuya palabra nunca vio: su vector es ruido
- Bolsa de palabras: «un 7 y no un 1» = «un 1 y no un 7»
- 1.200 imágenes y 4 plantillas, no 400 millones de pares
- Zero-shot ≠ magia: las clases tienen que estar en los textos

Seamos honestos con el juguete. Si una palabra nunca apareció en el entrenamiento, su embedding sigue siendo el inicial al azar; una clase nombrada así no puede funcionar, por construcción. El encoder de texto promedia palabras: no distingue un siete y no un uno de un uno y no un siete. CLIP usa un transformer para el texto, que sí mira el orden. Y son mil doscientas imágenes. Y el zero-shot de CLIP funciona porque, entre cuatrocientos millones de descripciones, casi cualquier clase que se te ocurra ya apareció escrita.

> **Radford et al., 2021 — CLIP, zero-shot en ImageNet** · [arXiv:2103.00020](https://arxiv.org/abs/2103.00020)
>
> **76,2 %** · Zero-shot en ImageNet: «matches the performance of the original ResNet-50 despite using none of the 1.28 million crowd-labeled training examples»; la plantilla «A photo of a {label}.» suma 1,3 %.

Y el resultado que hizo famoso a CLIP. Setenta y seis coma dos por ciento de precisión en ImageNet, zero-shot: lo mismo que la ResNet cincuenta original, sin usar ninguno de los un millón doscientos ochenta mil ejemplos etiquetados de ImageNet. Solo escribiendo los nombres de las mil clases. Y la plantilla a photo of a, una foto de un, suma uno coma tres puntos.

## Cierre · Lo que recorrimos

Diez episodios, de un tensor de píxeles a un modelo que une imágenes y texto.

### El mapa de la serie

El recorrido completo. Primero, la operación: una imagen es un tensor de canales por alto por ancho, la convolución es un kernel chico que se desliza, y en el fondo es una multiplicación de matrices con pesos compartidos. Después, la primera red: apilar convoluciones, ReLU, rectified linear unit, y pooling, y mirar qué aprende una CNN, convolutional neural network. Para entrenar redes hondas, la conexión residual y la BatchNorm, más data augmentation. Y al final, el cambio de idea: cortar la imagen en patches y usar atención, y unir imagen y texto con una pérdida contrastiva.

Imagen = tensor → Convolución → Conv = matriz → Canales y campo → LeNet → AlexNet → Qué mira una CNN → ResNet → BatchNorm y aug. → ViT → CLIP

### Las ideas que se repiten en todos lados

- Pesos compartidos: la conv, y la atención sobre tokens
- Sumar x: residual en ResNet y en cada transformer
- Normalizar: BatchNorm acá, layer norm en el ViT
- Un espacio de vectores: el coseno compara lo que sea

Cuatro ideas para llevarse. Compartir pesos: la convolución usa el mismo kernel en cada posición, y un transformer usa las mismas matrices en cada token. Sumar la entrada: y igual a x más F de x está en cada ResNet y en cada bloque de transformer. Normalizar las activaciones para que el entrenamiento no se desarme: BatchNorm en las convoluciones, layer norm en el transformer. Y convertir cualquier cosa en un vector, para que comparar sea un producto punto. Los modelos multimodales de hoy combinan estas piezas.

## Resumiendo

- CLIP: un encoder de imagen y otro de texto, al mismo espacio
- Matriz S = Î T̂ᵀ de N × N; la diagonal son los pares
- InfoNCE: entropía cruzada por filas y por columnas, / τ
- Zero-shot: una frase por clase es la última capa
- Juguete: 95,3 % con una plantilla nueva; CLIP: 76,2 % en ImageNet
- Próximo: correr los snippets y cambiarles una cosa